# Sandbox: FD strategies on a simple function

**Goal:** choose how Tesseract picks the FD step $\varepsilon$ ([#749](https://github.com/pasteurlabs/tesseract-core/issues/749))

In [396]:
import io
from math import comb

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from ipywidgets import (
    HTML,
    Dropdown,
    FloatLogSlider,
    FloatSlider,
    HBox,
    Image,
    Layout,
    SelectionSlider,
    VBox,
)
from pydantic import BaseModel

from tesseract_core.runtime import Array, Differentiable, Float64
from tesseract_core.runtime.experimental import finite_difference_jvp

np.seterr(over="ignore", invalid="ignore")  # a much too large step overflows cosh

{'divide': 'warn', 'over': 'ignore', 'under': 'ignore', 'invalid': 'ignore'}

# Part 1: one input

## 1. Test function

$$f(x) = A \cosh z \,\big(1 + \sigma\, \xi(x)\big), \quad z = \frac{x - c}{L}, \qquad f' = \frac{A}{L} \sinh z, \quad f''' = \frac{A}{L^3} \sinh z$$

- Knobs: input size $x_0$, scale $L$, noise $\sigma$
- $\xi \sim \mathcal{N}(0, 1)$ seeded by $x$: same input, same $f$ (like a solver)
- FD: Tesseract's `finite_difference_jvp`

In [397]:
class InputSchema(BaseModel):
    x: Differentiable[Array[(None,), Float64]]
    c: list[float]
    L: list[float]
    A: float = 1.0
    sigma: float = 0.0


class OutputSchema(BaseModel):
    f: Differentiable[Float64]


n_calls = 0  # every call to apply, whichever route it comes through


def apply(inputs: InputSchema) -> OutputSchema:
    global n_calls
    n_calls += 1
    x = np.asarray(inputs.x, dtype=float)
    f = inputs.A * np.cosh((x - inputs.c) / np.array(inputs.L)).sum()
    xi = np.random.default_rng(x.view(np.uint64)).standard_normal()
    return OutputSchema(f=f * (1 + inputs.sigma * xi))

Helpers: evaluation point, exact derivatives along $v$, Tesseract's FD along $v$ with step $t$.

In [398]:
def problem(x0=1.0, L=1.0, n=1, **knobs):
    """The point x and the knobs of f, with each input three length scales from its minimum."""
    x, L = np.broadcast_to(x0, n).astype(float), np.broadcast_to(L, n).astype(float)
    return x, {"c": list(x - 3 * L), "L": list(L), **knobs}


def f(x, knobs):
    return apply(InputSchema(x=x, **knobs)).f


def deriv(x, v, k, c, L, A=1.0, **noise):
    """The exact k-th derivative of f along v."""
    z = (x - np.array(c)) / L
    return A * np.sum(
        (v / np.array(L)) ** k * (np.cosh(z) if k % 2 == 0 else np.sinh(z))
    )


def grad(x, **knobs):
    return np.array([deriv(x, e, 1, **knobs) for e in np.eye(len(x))])


def fd(x, v, t, knobs):
    """Tesseract's central difference along v, with step t."""
    return finite_difference_jvp(
        apply, InputSchema(x=x, **knobs), {"x"}, {"f"}, {"x": v}, eps=t
    )["f"]

## 2. Where the FD error comes from

$$e(\varepsilon) \approx \underbrace{\frac{\varepsilon^2 |f'''|}{6 |f'|}}_\text{truncation} + \underbrace{\frac{\varepsilon_f}{\varepsilon |f'|}}_\text{noise}, \qquad \varepsilon^\star = \left( \frac{3\, \varepsilon_f}{|f'''|} \right)^{1/3}$$

**Best step needs two numbers: noise $\varepsilon_f$ and curvature $f'''$.**

<details>
<summary>Derivation</summary>

- Taylor, even terms cancel: $\frac{f(x_0 + \varepsilon) - f(x_0 - \varepsilon)}{2 \varepsilon} = f' + \frac{\varepsilon^2}{6} f''' + O(\varepsilon^4)$
- Noise: two calls, each off by $\le \varepsilon_f$, over $2\varepsilon$ → $\varepsilon_f / \varepsilon$
- Minimum at $\varepsilon^\star$; $e(\lambda \varepsilon^\star) = e^\star (\lambda^2 + 2/\lambda)/3 < 1.7\, e^\star$ for $\lambda \in [1/2, 2]$
- Here: $f''' = f'/L^2$, $\varepsilon_f = \sigma |f|$ + rounding of $f$ and $x_0$
</details>

Reference: exact $\varepsilon^\star$, from true noise and curvature.

In [399]:
u = np.finfo(float).eps / 2


def true_noise(x, v, knobs):
    """eps_f along v: the noise and rounding of f, and the rounding of the inputs v moves."""
    f_exact, g = deriv(x, np.ones(len(x)), 0, **knobs), grad(x, **knobs)
    return (knobs.get("sigma", 0) + u) * f_exact + u * np.abs(x * g) @ (v != 0)


def eps_star(x, v, knobs):
    """The best step along v, from the true noise and the exact third derivative."""
    return (3 * true_noise(x, v, knobs) / abs(deriv(x, v, 3, **knobs))) ** (1 / 3)

V-curve: measured error vs step, with truncation and noise terms.

In [400]:
def sliders():
    return {
        name: FloatLogSlider(
            value=value,
            min=lo,
            max=hi,
            step=step,
            description=name,
            continuous_update=False,
        )
        for name, value, lo, hi, step in (
            ("x0", 1.0, 0, 8, 0.5),
            ("L", 1.0, -3, 3, 0.25),
            ("sigma", 1e-8, -16, -4, 0.5),
        )
    }


redraws = []  # every live plot
updating = False  # holds them while many sliders move


def live(plot, controls, spectrum=False):
    """The controls, then a picture and a table that plot(**values) redraws in place when a control moves.

    With the spectrum of Part 2, plot also gets its L and sigma.
    The picture is an Image widget whose value is replaced, so a redraw can never add a second plot.
    """
    picture, table = Image(format="png"), HTML()

    def redraw(_=None):
        if updating:
            return
        values = {name: w.value for name, w in controls.items()}
        if spectrum:
            values |= {
                "L": 10.0 ** np.array([s.value for s in log_L]),
                "sigma": sigma_n.value,
            }
        fig, rows = plot(**values)
        buffer = io.BytesIO()
        fig.savefig(buffer, format="png", dpi=200, bbox_inches="tight")
        plt.close(fig)
        picture.value = buffer.getvalue()
        picture.width = str(int.from_bytes(picture.value[16:20], "big") // 2)  # 100 dpi
        table.value = f"<pre>{rows}</pre>" if rows else ""

    for w in [*controls.values(), *([*log_L, sigma_n] if spectrum else [])]:
        w.observe(redraw, names="value")
    redraws.append(redraw)
    redraw()
    return VBox([HBox(list(controls.values())), picture, table])


def v_curve(ax, x0, L, sigma):
    x, knobs = problem(x0, L, sigma=sigma)
    v, exact = np.ones(1), grad(x, **knobs)[0]
    eps = np.logspace(-14, 4, 145)
    eps = eps[
        (eps < 30 * L) & (eps > 2 * u * abs(x0))
    ]  # beyond: overflow, or x0 + eps == x0
    noise = true_noise(x, v, knobs) / abs(exact)
    ax.loglog(eps, [abs(fd(x, v, e, knobs) / exact - 1) for e in eps], label="measured")
    ax.loglog(eps, eps**2 / (6 * L**2), "k:", lw=0.8, label="truncation")
    ax.loglog(eps, noise / eps, "k--", lw=0.8, label="noise")
    ax.set(
        xlabel="eps", ylabel="relative error e", xlim=(1e-14, 1e4), ylim=(1e-14, 1e2)
    )
    return x, knobs, exact


def v_plot(x0, L, sigma):
    fig, ax = plt.subplots(figsize=(5, 3.2))
    v_curve(ax, x0, L, sigma)
    ax.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1.02, 1))
    return fig, ""


display(live(v_plot, sliders()))

**Flat bottom:** within ×2 of $\varepsilon^\star$ → error < 1.7 $e^\star$. Cube root → noise and curvature only need the right order of magnitude.

## 3. The methods

Estimate noise and curvature → $\varepsilon = (3 \hat\varepsilon_f / |\hat f'''|)^{1/3}$. Rescaling ([#516](https://github.com/pasteurlabs/tesseract-core/issues/516)) only moves a fixed $\varepsilon$: left out.

| Method | $\varepsilon$ | Calls | Ref |
|---|---|---|---|
| **fixed** | $10^{-4}$ | 2 | current default |
| **best-of-sweep** | $\{1, \dots, 10^{-4}\} \times \operatorname{rms}(x)$; midway between the two best-agreeing FD estimates | 12 | mosaic [`fd_check`](https://github.com/pasteurlabs/mosaic/blob/main/mosaic/benchmarks/problems/shared/gradient.py) |
| **adaptive** | noise: 9-call difference table; curvature: third difference at $h = 10^{-2} s$ | 15 | [Moré & Wild 2011](https://doi.org/10.1137/100786125) ([ECnoise](https://www.mcs.anl.gov/~wild/cnoise/)); [Gill et al. 1983](https://doi.org/10.1137/0904025) |
| **ratio test** | same noise; move $h$ (×2) until third difference is 30–300 × noise | 15–55 | [Shi et al. 2022](https://arxiv.org/abs/2110.06380) |

- Ratio test: tries spacings $h$ from $10^{-2} s$, doubling or halving until $f$'s curve shows above the noise; 4 calls per try; 30× because the third difference carries √10 × noise; applied to $f'''$ (Shi et al.: $f''$)
- Mosaic's `fd_check` picks the step that agrees with the gradient under test (passes if any step agrees); here: from $f$ alone

- Noise: ECnoise, 9 calls ([Moré & Wild](https://doi.org/10.1137/100786125): 7 for random noise, 9 for iterative solvers), good to ×4 → step to ×1.6 (here: 0.74–1.18 × the 20-call step, 90% of draws)
- Curvature: third difference, at a fixed spacing or with the ratio test

In [401]:
def ecnoise(x, v, knobs, h=1e-6, m=8, k=3):
    """Moré and Wild (2011): the noise of f from m + 1 calls along v, never below the rounding of f."""
    fs = np.array([f(x + j * h * v, knobs) for j in range(m + 1)])
    return max(
        np.sqrt(np.mean(np.diff(fs, k) ** 2) / comb(2 * k, k)), u * abs(fs.mean())
    )


def third_difference(x, v, h, knobs):
    fm2, fm1, fp1, fp2 = (f(x + j * h * v, knobs) for j in (-2, -1, 1, 2))
    return fp2 - 2 * fp1 + 2 * fm1 - fm2


def coarse(x, v, knobs, eps_f, h=1e-2):
    """Gill et al. (1983): the third derivative along v, at a fixed spacing h."""
    return third_difference(x, v, h, knobs) / (2 * h**3)


def ratio(x, v, knobs, eps_f, h=1e-2, r=(30, 300), tries=40):
    """Shi et al. (2022): the third derivative along v, at a spacing h where the third difference is r x the noise."""
    lo, hi = 0.0, np.inf  # the last spacings found too small and too large
    for _ in range(tries):
        d = third_difference(x, v, h, knobs)
        if not np.isfinite(d) or abs(d) > r[1] * eps_f:
            hi = h
        elif abs(d) < r[0] * eps_f:
            lo = h
        else:
            break
        h = 2 * h if hi == np.inf else h / 2 if lo == 0 else np.sqrt(lo * hi)
    return d / (2 * h**3)

In [402]:
def scale(x):
    return max(np.sqrt(np.mean(x**2)), 1)


def adaptive(curvature):
    """The step along unit v from a noise and a curvature estimate; the noise can be passed in, measured once."""

    def method(x, v, knobs, eps_f=None):
        s = scale(x)
        eps_f = ecnoise(x, s * v, knobs) if eps_f is None else eps_f
        d3 = abs(curvature(x, s * v, knobs, eps_f))
        if not np.isfinite(d3) or d3 == 0:
            return np.nan  # the spacing overflowed f
        return (3 * eps_f * s**3 / d3) ** (1 / 3)

    return method


mosaic_eps = np.array([1e0, 1e-1, 1e-2, 1e-3, 1e-4])  # mosaic's thermal-mesh configs


def best_of_sweep(x, v, knobs):
    """Mosaic's sweep, with the step chosen from f alone: between the consecutive FD estimates that agree best."""
    h = mosaic_eps * np.sqrt(np.mean(x**2))
    diffs = np.abs(np.diff([fd(x, v, hk, knobs) for hk in h]))
    if np.isnan(diffs).all():
        return np.nan  # every step of the sweep overflows f
    k = np.nanargmin(diffs)
    return np.sqrt(h[k] * h[k + 1])


methods = {
    "fixed": lambda x, v, knobs: 1e-4,
    "best-of-sweep": best_of_sweep,
    "adaptive": adaptive(coarse),
    "ratio test": adaptive(ratio),
}
styles = {
    "fixed": "oC1",
    "best-of-sweep": "sC2",
    "adaptive": "^C4",
    "ratio test": "vC5",
}

In [403]:
def methods_plot(x0, L, sigma):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.4))
    x, knobs, exact = v_curve(ax1, x0, L, sigma)
    v = np.ones(1)
    lines = ["        method |     eps | eps/eps* |   error | calls"]
    for name, method in methods.items():
        start = n_calls
        t = method(x, v, knobs)
        err = abs(fd(x, v, t, knobs) / exact - 1) if np.isfinite(t) else np.nan
        calls = n_calls - start
        ax1.plot(
            np.clip(t, 1e-14, 1e4),
            np.clip(err, 1e-14, 1e2),
            styles[name][0],
            ms=8,
            color=styles[name][1:],
            mfc="none",
            mew=1.5,
            clip_on=False,
            label=f"{name} ({calls} calls)",
        )
        if np.isfinite(t):
            lam = t / eps_star(x, v, knobs)
            lines.append(
                f"{name:>14} | {t:7.0e} | {lam:8.2g} | {err:7.0e} | {calls:5d}"
            )
        else:
            lines.append(f"{name:>14} | no step: its spacings overflow f | {calls:5d}")
    ax1.legend(fontsize=7, loc="lower left")

    Ls = np.logspace(-3, 3, 25)
    ax2.axhspan(0.5, 2, color="0.9")
    for name in ("best-of-sweep", "adaptive", "ratio test"):
        ratios = []
        for Lk in Ls:
            xk, kk = problem(x0, Lk, sigma=sigma)
            ratios.append(methods[name](xk, v, kk) / eps_star(xk, v, kk))
        ax2.loglog(
            Ls, ratios, "-" + styles[name][0], ms=4, color=styles[name][1:], label=name
        )
    ax2.axvline(L, color="0.5", lw=0.8)
    ax2.set(xlabel="L", ylabel="eps / eps*", ylim=(1e-4, 1e4))
    ax2.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1.02, 1))
    return fig, "\n".join(lines)


display(live(methods_plot, sliders()))

# Part 2: many inputs

## 4. Quick or thorough check

$$f(x) = A \sum_{i=1}^n \cosh\left( \frac{x_i - c_i}{L_i} \right) \big( 1 + \sigma\, \xi(x) \big)$$

FD along a direction $v$: $\frac{f(x + t v) - f(x - t v)}{2t} \approx v^\top g$. Step: ratio test; noise measured once (9 calls).

| | Directions | Calls | Guarantees |
|---|---|---|---|
| **thorough** | $e_i$, one per input | 9 + 6–50 per input | every component, to its FD error |
| **quick** | $k$ random | 9 + 10–60 per direction, any $n$ | a wrong gradient, if its error is a fair share of $g$ |

Cost per direction: 4 calls per try of the ratio test, starting at spacing $10^{-2}$; one more try per ×2 between that and the spacing where $f$'s curve shows (≈ $0.02\, L_i$ here). 

In [404]:
n = 16  # inputs: change and re-run for more sliders

presets = {  # log10 L per input: a quarter 3 decades up; spread over 3 decades; alike
    "two groups": [0] * (n - n // 4) + [3] * (n // 4),
    "even spread": [3 * i / (n - 1) for i in range(n)],
    "alike": [0] * n,
}
log_L = [
    FloatSlider(
        min=-3,
        max=6,
        step=0.5,
        readout_format=".1f",
        orientation="vertical",
        description=str(i),
        continuous_update=False,
        layout=Layout(width="44px"),
    )
    for i in range(n)
]
preset = Dropdown(options=list(presets), value="two groups", description="start from")
sigma_n = FloatLogSlider(
    value=1e-8, min=-16, max=-2, step=1, description="sigma", continuous_update=False
)


def set_preset(_=None):
    """Set every slider from the preset, then redraw once."""
    global updating
    updating = True
    for s, value in zip(log_L, presets[preset.value], strict=True):
        s.value = value
    updating = False
    for redraw in redraws:
        redraw()


preset.observe(set_preset, names="value")
set_preset()
spectrum = VBox(
    [HBox([preset, sigma_n]), HTML("log10 L per input"), HBox(log_L)],
    layout=Layout(border="1px solid lightgray", padding="4px"),
)
display(spectrum)

**Grading:** $g$ = exact gradient with one component wrong, $g_i \to (1 \pm \beta) g_i$, each $i$ in turn; identified if flagged for both signs. Exact values: grading only.

- Small $g_i$: hardest case for random directions
- Whole gradient wrong, $g \to (1 + \beta) g$: one direction of any kind finds it

### What one direction measures

One number, $v^\top g$, with FD error $e_v$. An error $\beta g_i$ shows if $\beta |g_i v_i| > e_v$.

1. **Weights:** $e_i$ → 1 on one input; random unit $v$ → about $1/\sqrt n$ on each ($\pm 1$, as in [SPSA](https://doi.org/10.1109/9.119632): exactly; Gaussian $z / \lVert z \rVert$: on average)
2. **Floor:** steep inputs set $e_v \approx e\, |v^\top g|$ → $g_i$ seen only if $|g_i| \gtrsim (e / \beta)\, \lVert g \rVert$. Along $e_i$: every component.
3. **Count:** $k < n$ directions = $k$ projections; more random directions repeat the same floor.

→ Random: the gradient as a whole. Coordinates: component by component.

Threshold per direction: 3 × predicted FD error + mismatch of the correct gradient.

In [405]:
def gaussian(k, n, rng):
    V = rng.standard_normal((k, n))
    return V / np.linalg.norm(V, axis=1, keepdims=True)


def calibrate(x, knobs, V, eps_f):
    """Per direction: what a component's contribution |g_i v_i| times the error must exceed, and the calls spent."""
    g, out = grad(x, **knobs), []
    for v in V:
        start = n_calls
        t = methods["ratio test"](x, v, knobs, eps_f)
        miss = abs(fd(x, v, t, knobs) - v @ g)
        tau = 3 * 1.5 * eps_f / t  # 3x the error the ratio test predicts
        out.append((np.inf if miss > tau else tau + miss, n_calls - start))
    return np.array(out).T

### 4.1 The price of each check

Chance to identify a component 5% wrong (mean over components) vs calls to `apply`.

- **Quick:** $k$ random directions (points); chance $1 - (1 - p_i)^k$, $p_i$ from 4000 directions
- **Thorough:** one direction per input (star); grey: stopped early
- **Calls:** 9 (noise) + per direction (4 per ratio-test try + 2); they move with the $L_i$

In [406]:
error = SelectionSlider(
    options=[1e-1, 5e-2, 1e-2, 1e-3],
    value=5e-2,
    description="error",
    continuous_update=False,
)


def price_plot(L, error, sigma, draws=4000):
    n = len(L)
    x, knobs = problem(L=L, n=n, sigma=sigma)
    rng, start = np.random.default_rng(0), n_calls
    eps_f = ecnoise(x, scale(x) * gaussian(1, n, rng)[0], knobs)
    noise_calls, g = n_calls - start, grad(x, **knobs)
    fig, ax = plt.subplots(figsize=(5.5, 3.4))
    ks = np.unique(np.logspace(0, np.log10(10 * n), 30).astype(int))
    for name, color, dy in (("gaussian", "C8", -11), ("+-1", "C9", 6)):
        if name == "+-1":
            V = rng.choice([-1.0, 1.0], (draws, n)) / np.sqrt(n)
        else:
            V = gaussian(draws, n, rng)
        theta, calls = calibrate(x, knobs, V, eps_f)
        p = (error * np.abs(g * V) > theta[:, None]).mean(axis=0)  # per component
        cost = noise_calls + ks * calls.mean()
        share = [np.mean(1 - (1 - p) ** k) for k in ks]
        ax.semilogx(cost, share, color=color, lw=2, label=f"quick, {name}")
        for k in (1, 4, 16, 64):  # the choice: how many directions to check
            point = (noise_calls + k * calls.mean(), np.mean(1 - (1 - p) ** k))
            ax.plot(*point, "o", ms=4, color=color)
            ax.annotate(
                f"k={k}",
                point,
                textcoords="offset points",
                xytext=(0, dy),
                fontsize=7,
                ha="center",
                color=color,
            )
    theta, calls = calibrate(x, knobs, np.eye(n), eps_f)
    caught, total = np.mean(error * np.abs(g) > theta), noise_calls + calls.sum()
    ax.semilogx(
        noise_calls + np.cumsum(calls),
        np.arange(1, n + 1) / n * caught,
        color="C7",
        lw=2,
        label="coordinates",
    )
    ax.plot(total, caught, "*", ms=12, color="k", label=f"thorough ({total:.0f} calls)")
    ax.set(
        xlabel="calls to apply",
        ylabel=f"chance to identify a {error:g} error",
        ylim=(-0.03, 1.03),
    )
    ax.xaxis.set_minor_formatter(plt.NullFormatter())
    ax.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1.02, 1))
    return fig, ""


display(spectrum, live(price_plot, {"error": error}, spectrum=True))

**Results** (one component 5% wrong):

| Preset | quick, 1 direction | quick, 16 directions | thorough |
|---|---|---|---|
| alike | 100%, 22–31 calls | 100% | 161 calls |
| two groups | 75% Gaussian, 80% $\pm 1$, 22–31 calls | 83% Gaussian (221 calls), 100% $\pm 1$ (357) | 313 calls |
| even spread | 83% Gaussian, 89% $\pm 1$, 26–27 calls | 100% Gaussian (293 calls), 94% $\pm 1$ (288) | 469 calls |

- 1 random direction: most components, 22–31 calls, any $n$
- Quick checks stop at the floor: components with less than a share $e/\beta$ of $g$
- $\pm 1$ lead on two groups: artefact (12 equal steep inputs balance 6/6 in 23% of directions → curvatures cancel → larger step); jitter $L_i$ by 0.3 decades → both 75–81%
- Quick at 100% ≈ cost of thorough, without its guarantee
- Only thorough guarantees every component: 160–470 calls here

## Conclusion

- **Step:** ECnoise (once) + ratio test → within ×2 of best, from $f$ alone, 15–55 calls; same step for FD gradient and check
- **Thorough check (= FD gradient):** one difference per $e_i$, own step, tolerance 3 × predicted error; 9 + 6–50 calls per input (160–470 for $n = 16$); catches any component wrong beyond its FD error (~$10^{-5}$)
- **Quick check:** same along $k = 1$–4 random directions (Gaussian ≈ $\pm 1$); 20–100 calls, any $n$; finds a gradient wrong as a whole, a 5% component error in 75–100% of components; can't vouch for small $|g_i|$; no FD gradient

### What's next

Holds for a smooth $f$ with independent noise; to measure on a real solver.

- **Optimization:** curvature changes along the path, $f''' \to 0$ near a minimum; rerun the ratio test when iterates move far ([Berahas et al. 2019](https://arxiv.org/abs/1803.10173), [Shi et al. 2022](https://arxiv.org/abs/2010.04352))
- **Noise:** correlated (warm starts, tolerances), input-dependent (remeshing)
- **Function:** kinks, discontinuities, coupled inputs, $f''' \approx 0$, vector outputs
- **Quick check:** report which components it can vouch for (from $|g_i v_i|$)
- **Solver:** the physical solver of `demo.ipynb`; per-element steps in `check_gradients`